# 5장 결정 트리와 앙상블 — 지니 불순도, 가지치기, 특징 중요도, 랜덤 포레스트, 그레이디언트 부스팅
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 펭귄 종을 맞히는 결정 트리를 만들고 읽어 보기
2. 지니 불순도 직접 계산하기
3. 깊이를 바꾸며 과적합 확인하고 교차 검증으로 깊이 고르기
4. 특징 중요도 읽기
5. 유방암 진단 데이터로 랜덤 포레스트와 그레이디언트 부스팅 비교하기

위에서부터 차례로 실행하세요.

## 1. 결정 트리 만들기
2장처럼 부리 길이와 날개 길이로 펭귄의 종을 맞힙니다. 질문을 두 번까지만 하도록 `max_depth=2`로 제한합니다. `weights`는 그 자리에 도착한 학습 데이터 펭귄 수(아델리, 턱끈, 젠투 순서)입니다.

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

df = sns.load_dataset("penguins").dropna(
    subset=["bill_length_mm", "flipper_length_mm"])
X = df[["bill_length_mm", "flipper_length_mm"]]
y = df["species"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(Xtr, ytr)
print(export_text(tree, feature_names=["부리 길이", "날개 길이"],
                  show_weights=True))

### 스케일링이 필요 없다
표준화한 데이터로 학습해도 예측이 같은지 확인합니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plain = DecisionTreeClassifier(max_depth=3, random_state=0)
scaled = make_pipeline(
    StandardScaler(), DecisionTreeClassifier(max_depth=3, random_state=0))
p1 = plain.fit(Xtr, ytr).predict(Xte)
p2 = scaled.fit(Xtr, ytr).predict(Xte)
print("예측이 모두 같은가:", (p1 == p2).all())

## 2. 지니 불순도
무리에서 한 마리를 뽑아 종을 보고 다시 넣은 뒤 또 뽑았을 때 두 종이 다를 확률 = 1 − (각 종 비율의 제곱의 합). 첫 질문 전후의 세 무리로 계산합니다.

In [ ]:
def gini(counts):
    p = np.array(counts) / sum(counts)    # 종마다의 비율
    return 1 - (p ** 2).sum()

print(f"질문 전 : {gini([113, 51, 92]):.3f}")
print(f"왼쪽 무리: {gini([111, 46, 1]):.3f}")
print(f"오른쪽 무리: {gini([2, 5, 91]):.3f}")

## 3. 과적합과 가지치기
깊이를 제한하지 않으면 학습 데이터를 100% 맞히지만 테스트 정확도는 오히려 낮아집니다.

In [ ]:
for depth in [2, None]:                 # None은 제한 없음
    t = DecisionTreeClassifier(max_depth=depth, random_state=0)
    t.fit(Xtr, ytr)
    print(f"max_depth={depth}: 깊이 {t.get_depth()}, "
          f"잎 {t.get_n_leaves()}개, 학습 {t.score(Xtr, ytr):.1%}, "
          f"테스트 {t.score(Xte, yte):.1%}")

깊이를 1부터 10까지 바꾸며 5-폴드 교차 검증으로 비교합니다.

In [ ]:
from sklearn.model_selection import cross_val_score

for depth in range(1, 11):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv = cross_val_score(t, Xtr, ytr, cv=5)
    print(f"깊이 {depth:2d}: 교차 검증 정확도 {cv.mean():.1%}"
          f" (표준편차 {cv.std():.1%})")

## 4. 특징 중요도
측정값 네 개를 모두 주고 깊이 3인 트리를 학습합니다. 중요도가 낮다고 쓸모없는 특징은 아닙니다(비슷한 정보를 다른 특징이 먼저 썼을 수 있음).

In [ ]:
num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
df4 = sns.load_dataset("penguins").dropna(subset=num)
A, B, a, b = train_test_split(
    df4[num], df4["species"], test_size=0.25,
    random_state=42, stratify=df4["species"])
t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(A, a)
for name, imp in zip(num, t.feature_importances_):
    print(f"{name:18s} {imp:.3f}")
print(f"테스트 정확도: {t.score(B, b):.1%}")

## 5. 결정 트리는 흔들린다
4장의 유방암 진단 데이터에서 부트스트랩(중복을 허락해 다시 뽑기)으로 만든 다섯 학습 데이터로 트리를 학습합니다. 첫 질문과 답이 얼마나 달라지는지 보세요.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X = cancer.data
y = (cancer.target == 0).astype(int)      # 악성이면 1
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

rng = np.random.default_rng(0)
preds = []
for i in range(5):
    # 부트스트랩: 같은 개수만큼 다시 뽑되, 중복을 허락한다
    idx = rng.choice(len(Xtr), size=len(Xtr), replace=True)
    t = DecisionTreeClassifier(random_state=0)
    t.fit(Xtr.iloc[idx], ytr.iloc[idx])
    root = X.columns[t.tree_.feature[0]]     # 첫 질문의 특징 이름
    print(f"나무 {i+1}: 서로 다른 샘플 {len(set(idx))}개, "
          f"첫 질문 '{root}', 테스트 {t.score(Xte, yte):.1%}")
    preds.append(t.predict(Xte))
P = np.array(preds)
diff = (P.min(axis=0) != P.max(axis=0)).sum()  # 답이 엇갈린 샘플 수
print("다섯 나무의 답이 엇갈린 테스트 샘플:", diff)

## 6. 랜덤 포레스트
부트스트랩 데이터와 무작위로 고른 특징으로 키운 나무 100그루의 의견을 모읍니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

single = DecisionTreeClassifier(random_state=0)
forest = RandomForestClassifier(random_state=0)     # 나무 100그루
for name, m in [("결정 트리", single), ("랜덤 포레스트", forest)]:
    cv = cross_val_score(m, Xtr, ytr, cv=5)
    print(f"{name}: 교차 검증 정확도 {cv.mean():.1%}"
          f" (표준편차 {cv.std():.1%})")

## 7. 그레이디언트 부스팅
작은 나무를 하나씩 더하며 앞 나무들이 틀린 만큼을 고칩니다. 나무 수에 따라 정확도가 어떻게 변하는지 봅니다.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=0)    # 나무 100그루
gb.fit(Xtr, ytr)
tr = [(p == ytr).mean() for p in gb.staged_predict(Xtr)]
te = [(p == yte).mean() for p in gb.staged_predict(Xte)]
for n in [1, 5, 10, 20, 50, 100]:
    print(f"나무 {n:3d}그루: "
          f"학습 {tr[n-1]:.1%}, 테스트 {te[n-1]:.1%}")

## 8. 모델 비교
4장의 k-최근접 이웃과 이 장의 세 모델을 같은 5-폴드 교차 검증으로 비교합니다 (조금 시간이 걸립니다).

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier

models = {
    "k-최근접 이웃": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "결정 트리": DecisionTreeClassifier(random_state=0),
    "랜덤 포레스트": RandomForestClassifier(random_state=0),
    "그레이디언트 부스팅":
        GradientBoostingClassifier(random_state=0),
}
for name, m in models.items():
    cv = cross_validate(m, Xtr, ytr, cv=5,
                        scoring=["accuracy", "f1"])
    acc, f1 = cv["test_accuracy"], cv["test_f1"]
    print(f"{name}: 정확도 {acc.mean():.1%} (±{acc.std():.1%}), "
          f"F1 {f1.mean():.1%}")

---
## 실습 문제

**문제 1.** 이 노트북 1절(이북 5.1절)의 깊이 2 트리를 `criterion="entropy"`로 바꾸어 학습하고 `export_text`로 출력해 보세요. 지니 불순도로 만든 트리와 무엇이 같고 무엇이 다른가요? 테스트 정확도는 어떤가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 이 노트북 3절(이북 5.3절)에서 깊이 대신 `min_samples_leaf`를 1, 2, 5, 10, 20으로 바꾸어 5-폴드 교차 검증 정확도를 비교해 보세요(깊이는 제한하지 않음). 각 값에서 잎의 수(`get_n_leaves()`)도 확인해 보세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 이 노트북 6절(이북 5.5절)의 랜덤 포레스트에서 나무 수(`n_estimators`)를 1, 10, 50, 100, 300으로 바꾸어 5-폴드 교차 검증 정확도를 비교해 보세요. 나무가 많아질수록 어떻게 되나요?

In [ ]:
# 문제 3 풀이

**문제 4.** 그레이디언트 부스팅의 학습률(`learning_rate`)을 0.01, 0.1, 0.5로 바꾸어 5-폴드 교차 검증 정확도를 비교해 보세요(나무 수는 기본값 100). 학습률 0.01의 결과가 낮은 이유는 무엇일까요?

In [ ]:
# 문제 4 풀이